# Granite Peak CRE: 05 Tenant renewal-propensity model

Predicts the probability that each active lease renews at expiry, then prices the risk:

`expected_vacancy_loss = (1 − renewal_probability) × ABR × 9 / 12`, assuming nine months of downtime and re-leasing time when a tenant leaves.

**Features** combine all three sources in the build:
- ERP: credit rating, payment history, rent versus market, tenure.
- Lease documents (`ai_extract`): renewal options, co-tenancy, termination right.
- H3 context (Overture): competitor and amenity density within about 1 km.

**Training labels are synthetic.** Granite Peak has no real renewal history. The notebook samples 1,500 historical lease expirations over the real properties and their H3 context, and draws each outcome from a documented logistic function (cell 3). The model is expected to recover those drivers; its feature importances are checked against them.

The model is tracked in MLflow, registered in Unity Catalog as `renewal_propensity`, and batch-scores active leases into `renewal_scores`.

In [1]:
%pip install -q "mlflow>=3" scikit-learn
dbutils.library.restartPython()

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [2]:
dbutils.widgets.text("catalog", "smmorgan_catalog", "Catalog")
dbutils.widgets.text("schema", "granite_commercial_real_estate", "Schema")
catalog = dbutils.widgets.get("catalog").strip()
schema = dbutils.widgets.get("schema").strip()
uc = f"{catalog}.{schema}"
MODEL_NAME = f"{uc}.renewal_propensity"
DOWNTIME_MONTHS = 9
print(f"Target: {uc}  |  Model: {MODEL_NAME}")

Target: smmorgan_catalog.granite_commercial_real_estate  |  Model: smmorgan_catalog.granite_commercial_real_estate.renewal_propensity


In [3]:
import numpy as np, pandas as pd

CREDIT_SCORE = {"AAA": 6, "AA": 5, "A": 4, "BBB": 3, "BB": 2, "B": 1}

# Property-level context shared by history and scoring
props = spark.sql(f"""
SELECT p.property_id, p.property_type, p.msa, m.competitor_count, m.amenity_count
FROM {uc}.property_h3 p JOIN {uc}.h3_market_context m USING (property_id)
""").toPandas()
props["competitor_density"] = props.groupby("property_type")["competitor_count"].transform(
    lambda s: (s - s.mean()) / (s.std() + 1e-9))
props["amenity_density"] = (props["amenity_count"] - props["amenity_count"].mean()) / (props["amenity_count"].std() + 1e-9)
print(props.describe().round(2).to_string())

       competitor_count  amenity_count  competitor_density  amenity_density
count            118.00         118.00              118.00           118.00
mean             219.58         146.78               -0.00             0.00
std              454.16         181.43                0.99             1.00
min                0.00           0.00               -1.18            -0.81
25%               16.25          27.25               -0.58            -0.66
50%               49.50          66.00               -0.38            -0.45
75%              183.25         196.00                0.25             0.27
max             2873.00         902.00                4.43             4.16


In [4]:
# Synthetic renewal history: the documented label function
rng = np.random.default_rng(7)
n = 1500
hist = props.sample(n, replace=True, random_state=7).reset_index(drop=True)
hist["credit_score"]         = rng.choice(list(CREDIT_SCORE.values()), n, p=[.05, .10, .25, .30, .20, .10])
hist["late_payment_ratio"]   = np.clip(rng.beta(1.2, 18, n), 0, 1)
hist["renewal_option_count"] = rng.choice([0, 0, 1, 1, 2, 3], n)
hist["has_co_tenancy"]       = (rng.random(n) < np.where(hist.property_type == "Retail", .25, .05)).astype(int)
hist["has_termination_right"] = (rng.random(n) < .2).astype(int)
hist["rent_to_market"]       = rng.normal(1.0, .12, n)
hist["tenure_years"]         = rng.uniform(1, 15, n)

logit = (0.3
         + 0.35 * (hist.credit_score - 3.5)
         - 6.0  * hist.late_payment_ratio
         + 0.40 * hist.renewal_option_count
         - 0.60 * hist.has_co_tenancy
         - 0.50 * hist.has_termination_right
         - 3.0  * (hist.rent_to_market - 1.0)
         + 0.06 * hist.tenure_years
         - 0.45 * hist.competitor_density
         + 0.20 * hist.amenity_density)
hist["renewed"] = (rng.random(n) < 1 / (1 + np.exp(-logit))).astype(int)
print(f"Synthetic expirations: {n:,}  |  renewal rate: {hist.renewed.mean():.1%}")
spark.createDataFrame(hist).write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{uc}.ml_renewal_history")

Synthetic expirations: 1,500  |  renewal rate: 61.2%


In [5]:
import mlflow
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.inspection import permutation_importance
from sklearn.metrics import roc_auc_score, brier_score_loss
from sklearn.model_selection import train_test_split

FEATURES = ["credit_score", "late_payment_ratio", "renewal_option_count", "has_co_tenancy",
            "has_termination_right", "rent_to_market", "tenure_years", "competitor_density", "amenity_density"]
X_tr, X_te, y_tr, y_te = train_test_split(hist[FEATURES], hist.renewed, test_size=0.25, random_state=7, stratify=hist.renewed)

mlflow.set_registry_uri("databricks-uc")
mlflow.set_experiment(f"/Users/{spark.sql('SELECT current_user()').first()[0]}/granite_renewal_propensity")
with mlflow.start_run(run_name="hgb_renewal_propensity") as run:
    params = {"max_depth": 3, "learning_rate": 0.05, "max_iter": 300, "random_state": 7}
    model = HistGradientBoostingClassifier(**params).fit(X_tr, y_tr)
    p_te = model.predict_proba(X_te)[:, 1]
    metrics = {"test_auc": roc_auc_score(y_te, p_te), "test_brier": brier_score_loss(y_te, p_te),
               "train_rows": len(X_tr), "test_rows": len(X_te), "base_renewal_rate": float(hist.renewed.mean())}
    mlflow.log_params(params); mlflow.log_metrics(metrics)
    imp = permutation_importance(model, X_te, y_te, scoring="roc_auc", n_repeats=10, random_state=7)
    importance = pd.Series(imp.importances_mean, index=FEATURES).sort_values(ascending=False)
    mlflow.log_dict(importance.round(4).to_dict(), "permutation_importance.json")
    info = mlflow.sklearn.log_model(model, name="model", input_example=X_te.head(5),
                                    registered_model_name=MODEL_NAME)
print({k: round(v, 4) if isinstance(v, float) else v for k, v in metrics.items()})
print("\nPermutation importance (drop in AUC):")
print(importance.round(4).to_string())
print(f"\nRegistered {MODEL_NAME} version {info.registered_model_version}")

If you are using MLflow Tracing, you can migrate your traces to Unity Catalog for unlimited storage, fine-grained access controls, and queryability from notebooks, SQL, and dashboards. Learn more: https://docs.databricks.com/aws/en/mlflow3/genai/tracing/migrate-traces-to-uc
🔗 View Logged Model at: <workspace-url>
/local_disk0/.ephemeral_nfs/envs/pythonEnv-120ea7be-b3c6-4022-bbfe-97d51505e41c/lib/python3.12/site-packages/mlflow/types/utils.py:440: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Mis

{"text/plain": "Uploading artifacts:   0%|          | 0/11 [00:00<?, ?it/s]", "application/vnd.jupyter.widget-view+json": {"version_major": 2, "version_minor": 0, "model_id": "8a18c69e41a748619daeefee5ea720ab"}}

🔗 Created version '3' of model 'smmorgan_catalog.granite_commercial_real_estate.renewal_propensity': <workspace-url>


{'test_auc': 0.7147, 'test_brier': 0.212, 'train_rows': 1125, 'test_rows': 375, 'base_renewal_rate': 0.612}

Permutation importance (drop in AUC):
credit_score             0.0686
rent_to_market           0.0496
renewal_option_count     0.0462
competitor_density       0.0347
tenure_years             0.0287
late_payment_ratio       0.0187
has_co_tenancy           0.0087
amenity_density          0.0062
has_termination_right    0.0047

Registered smmorgan_catalog.granite_commercial_real_estate.renewal_propensity version 3


In [6]:
# Score active leases: ERP + payment history + document clauses + H3 context
active = spark.sql(f"""
WITH pay AS (
  SELECT lease_id, avg(CASE WHEN paid THEN 0.0 ELSE 1.0 END) AS late_payment_ratio
  FROM {uc}.silver_lease_charges GROUP BY lease_id
),
mkt AS (
  SELECT msa, property_type, percentile(annual_base_rent / unit_sqft, 0.5) AS market_psf
  FROM {uc}.rent_roll GROUP BY ALL
)
SELECT r.lease_id, r.property_id, r.tenant_name, r.fund_id, r.property_type, r.msa, r.lease_end,
       r.months_to_expiry, r.annual_base_rent, r.credit_rating, r.has_abstract,
       coalesce(p.late_payment_ratio, 0) AS late_payment_ratio,
       coalesce(r.renewal_option_count, 0) AS renewal_option_count,
       CAST(coalesce(r.has_co_tenancy, false) AS INT) AS has_co_tenancy,
       CAST(coalesce(r.has_termination_right, false) AS INT) AS has_termination_right,
       (r.annual_base_rent / r.unit_sqft) / m.market_psf AS rent_to_market,
       datediff(r.lease_end, r.lease_start) / 365.25 AS tenure_years
FROM {uc}.rent_roll r
LEFT JOIN pay p USING (lease_id)
JOIN mkt m ON m.msa = r.msa AND m.property_type = r.property_type
""").toPandas().merge(props[["property_id", "competitor_density", "amenity_density"]], on="property_id")
active["credit_score"] = active.credit_rating.map(CREDIT_SCORE)

scorer = mlflow.sklearn.load_model(f"models:/{MODEL_NAME}/{info.registered_model_version}")
active["renewal_probability"] = scorer.predict_proba(active[FEATURES])[:, 1].round(4)
active["expected_vacancy_loss"] = ((1 - active.renewal_probability) * active.annual_base_rent * DOWNTIME_MONTHS / 12).round(2)
active["model_version"] = str(info.registered_model_version)

out = active[["lease_id", "property_id", "tenant_name", "fund_id", "property_type", "msa", "lease_end",
              "months_to_expiry", "annual_base_rent", "has_abstract", "renewal_probability",
              "expected_vacancy_loss", "model_version"]]
spark.createDataFrame(out).write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{uc}.renewal_scores")
spark.sql(f"COMMENT ON TABLE {uc}.renewal_scores IS 'Renewal probability and expected vacancy loss per active lease (model {MODEL_NAME}); vacancy loss assumes {DOWNTIME_MONTHS} months of downtime'")
print(f"Scored {len(out):,} active leases into {uc}.renewal_scores")

{"text/plain": "Downloading artifacts:   0%|          | 0/11 [00:00<?, ?it/s]", "application/vnd.jupyter.widget-view+json": {"version_major": 2, "version_minor": 0, "model_id": "079555cc0a714b7c8716e6df0eb51096"}}

Scored 391 active leases into smmorgan_catalog.granite_commercial_real_estate.renewal_scores


In [7]:
display(spark.sql(f"""
SELECT fund_id,
       count(*) AS leases,
       round(avg(renewal_probability), 3) AS avg_renewal_probability,
       round(sum(expected_vacancy_loss)) AS expected_vacancy_loss,
       round(sum(expected_vacancy_loss) FILTER (WHERE months_to_expiry <= 24)) AS expected_vacancy_loss_24m
FROM {uc}.renewal_scores GROUP BY fund_id ORDER BY fund_id
"""))
display(spark.sql(f"""
SELECT lease_id, tenant_name, msa, property_type, lease_end, round(annual_base_rent) AS abr,
       renewal_probability, round(expected_vacancy_loss) AS expected_vacancy_loss
FROM {uc}.renewal_scores WHERE months_to_expiry <= 24
ORDER BY expected_vacancy_loss DESC LIMIT 10
"""))

fund_id    | leases | avg_renewal_probability | expected_vacancy_loss | expected_vacancy_loss_24m
-----------+--------+-------------------------+-----------------------+--------------------------
GP-FND-001 | 80     | 0.544                   | 27325130.0            | 8801651.0                
GP-FND-002 | 106    | 0.539                   | 42807970.0            | 9441970.0                
GP-FND-003 | 94     | 0.54                    | 32273312.0            | 10123002.0               
GP-FND-004 | 111    | 0.533                   | 40482705.0            | 6159336.0                


lease_id     | tenant_name           | msa            | property_type | lease_end  | abr       | renewal_probability | expected_vacancy_loss
-------------+-----------------------+----------------+---------------+------------+-----------+---------------------+----------------------
GP-LSE-00150 | Mesa Manufacturing    | Phoenix        | Industrial    | 2027-01-30 | 5606760.0 | 0.035               | 4057893.0            
GP-LSE-00278 | Mesa Legal Group      | Las Vegas      | Office        | 2027-07-08 | 3534773.0 | 0.0517              | 2514019.0            
GP-LSE-00077 | Falcon Advisors       | Denver         | Office        | 2027-07-25 | 2480637.0 | 0.1093              | 1657127.0            
GP-LSE-00338 | Silver Partners       | Seattle        | Office        | 2027-03-21 | 2581028.0 | 0.2079              | 1533324.0            
GP-LSE-00023 | Summit Materials      | Salt Lake City | Industrial    | 2028-04-11 | 4196163.0 | 0.529               | 1482295.0            
GP-LSE-00129 